# Point-vortex configurations
Plot one or more saved vortex configurations. Geometry and domain dimensions are read from the run record by default, with explicit overrides available in the configuration cell.

In [ ]:
from pathlib import Path
import math
import sys

import matplotlib.pyplot as plt
import numpy as np

SCRIPT_DIRECTORY = Path.cwd() if (Path.cwd() / 'point_vortex_plotting.py').exists() else Path.cwd() / 'scripts'
sys.path.insert(0, str(SCRIPT_DIRECTORY.resolve()))
from point_vortex_plotting import (axis_limits, display_coordinates, draw_boundary,
    evenly_spaced_frames, marker_areas, read_parameters, read_trajectory,
    repository_root, resolve_domain, save_figure, select_frames, use_plot_style)

## Configuration

In [ ]:
ROOT = repository_root()  # Repository root; normally no change is needed.
RUN_DIRECTORY = ROOT / 'runs/default'  # Completed solver run to visualize.
TRAJECTORY_FILE = RUN_DIRECTORY / 'trajectory.csv'
PARAMETER_FILE = RUN_DIRECTORY / 'resolved_parameters.txt'
CONFIGURATION_FIGURE = RUN_DIRECTORY / 'figures/vortices.pdf'

# Exact frame labels to plot. Negative indices count from the end, so [-1]
# selects the newest frame. Set to None to use START/STOP/STRIDE and then
# choose up to SNAPSHOT_COUNT evenly spaced configurations from that range.
FRAMES = [-1]
FRAME_START = None  # First frame label when FRAMES=None; None means first.
FRAME_STOP = None   # Last frame label, inclusive; None means last.
FRAME_STRIDE = 1    # Keep every nth available frame in the selected range.
SNAPSHOT_COUNT = 4  # Maximum panels when FRAMES=None.

# Geometry is normally inferred from PARAMETER_FILE. Set GEOMETRY to
# 'infinite', 'periodic_x', 'periodic', or 'disk' only when an explicit override is wanted.
GEOMETRY = None
# For a square periodic display, set BOX_LENGTH and leave BOX_LENGTH_X/Y None.
# For a rectangular periodic display, leave BOX_LENGTH None and set X and Y.
BOX_LENGTH = None
BOX_LENGTH_X = None
BOX_LENGTH_Y = None
DISK_RADIUS = None  # Disk-boundary radius; None reads it from the run record.
X_LIMITS = None     # Explicit (minimum, maximum), or None for geometry defaults.
Y_LIMITS = None

PANEL_COLUMNS = 2
MARKER_AREA_MIN = 18.0
MARKER_AREA_MAX = 40.0
FIGURE_TITLE = None  # None creates a geometry-aware title.
USE_TEX = True       # False avoids requiring an external LaTeX installation.
FONT_SIZE = 14

## Load and validate the selected configurations

In [ ]:
if FRAME_STRIDE < 1 or SNAPSHOT_COUNT < 1 or PANEL_COLUMNS < 1:
    raise ValueError('FRAME_STRIDE, SNAPSHOT_COUNT, and PANEL_COLUMNS must be positive')
if not (0.0 < MARKER_AREA_MIN <= MARKER_AREA_MAX):
    raise ValueError('marker areas require 0 < MARKER_AREA_MIN <= MARKER_AREA_MAX')

use_plot_style(USE_TEX, FONT_SIZE)
parameters = read_parameters(PARAMETER_FILE) if PARAMETER_FILE.exists() else {}
domain = resolve_domain(parameters, geometry=GEOMETRY, box_length=BOX_LENGTH,
                        box_length_x=BOX_LENGTH_X, box_length_y=BOX_LENGTH_Y,
                        disk_radius=DISK_RADIUS, x_limits=X_LIMITS, y_limits=Y_LIMITS)
trajectory = read_trajectory(TRAJECTORY_FILE)
if FRAMES is None:
    candidates = select_frames(trajectory, start=FRAME_START, stop=FRAME_STOP,
                               stride=FRAME_STRIDE)
    frames = evenly_spaced_frames(candidates, SNAPSHOT_COUNT)
else:
    frames = select_frames(trajectory, FRAMES)
configurations = [trajectory[frame] for frame in frames]
x_limits, y_limits = axis_limits(configurations, domain)
global_strength = max((float(np.max(np.abs(frame.circulation)))
                       for frame in configurations), default=0.0)
print(f'Geometry: {domain.geometry}; selected frames: {frames}')

## Vortex configuration figure

In [ ]:
columns = min(PANEL_COLUMNS, len(configurations))
rows = math.ceil(len(configurations) / columns)
fig, axes_array = plt.subplots(rows, columns, figsize=(5.0 * columns, 4.6 * rows),
                                squeeze=False)
axes = list(axes_array.flat)
for axis, frame in zip(axes, configurations):
    x, y = display_coordinates(frame, domain)
    gamma = frame.circulation
    sizes = marker_areas(gamma, MARKER_AREA_MIN, MARKER_AREA_MAX,
                         scale=global_strength)
    draw_boundary(axis, domain)
    axis.scatter(x[gamma > 0.0], y[gamma > 0.0], s=sizes[gamma > 0.0],
                 color='tab:red', label=r'$\Gamma>0$', alpha=0.85)
    axis.scatter(x[gamma < 0.0], y[gamma < 0.0], s=sizes[gamma < 0.0],
                 color='tab:blue', label=r'$\Gamma<0$', alpha=0.85)
    axis.scatter(x[gamma == 0.0], y[gamma == 0.0], s=sizes[gamma == 0.0],
                 color='0.5', label=r'$\Gamma=0$', alpha=0.85)
    axis.set(xlim=x_limits, ylim=y_limits, xlabel=r'$x$', ylabel=r'$y$',
             title=rf'frame {frame.frame}, $t={frame.time:.6g}$, $N={frame.x.size}$')
    axis.set_aspect('equal', adjustable='box')
for axis in axes[len(configurations):]:
    axis.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
category_present = (
    any(np.any(frame.circulation > 0.0) for frame in configurations),
    any(np.any(frame.circulation < 0.0) for frame in configurations),
    any(np.any(frame.circulation == 0.0) for frame in configurations),
)
visible = [(handle, label) for handle, label, present
           in zip(handles, labels, category_present) if present]
if visible:
    fig.legend([entry[0] for entry in visible], [entry[1] for entry in visible],
               loc='outside right upper', frameon=False)
fig.suptitle(FIGURE_TITLE or f'Point-vortex configurations: {domain.geometry} geometry')
saved = save_figure(fig, CONFIGURATION_FIGURE)
print(f'Wrote {saved}')
plt.show()